In [1]:
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import TensorDataset, DataLoader

In [2]:
X = torch.randn(1000, 4)
y = 2*X[:, 0:1] + 0.5*X[:, 1:2] - 1.5*X[:, 2:3]**2 + X[:, 3:4]*X[:, 1:2] + 1
print("X size: ", X.shape)
print("y size: ", y.shape)

X size:  torch.Size([1000, 4])
y size:  torch.Size([1000, 1])


In [3]:
train_size = int(0.7 * len(X))
val_size = int(0.15 * len(X))

X_train = X[:train_size]
y_train = y[:train_size]

X_val = X[train_size:train_size + val_size]
y_val = y[train_size:train_size + val_size]

X_test = X[train_size + val_size:]
y_test = y[train_size + val_size:]

print("X_train:", X_train.shape)
print("X_val:", X_val.shape)
print("X_test:", X_test.shape)

X_train: torch.Size([700, 4])
X_val: torch.Size([150, 4])
X_test: torch.Size([150, 4])


In [4]:
train_dataset = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_dataset, batch_size = 32, shuffle = True)

In [5]:
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = nn.Linear(4, 60)
        self.relu = nn.ReLU()
        self.output = nn.Linear(60, 1)

    def forward(self, x):
        x = self.hidden(x)
        x = self.relu(x)
        x = self.output(x)
        return x

model = NeuralNetwork()
criterion = nn.MSELoss() 
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

In [6]:
epochs = 200
for epoch in range(epochs):
    model.train()  # явно включаем режим обучения
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad() #Обнуляет градиенты, накопленные с предыдущего шага
        output = model(X_batch)
        loss = criterion(output, y_batch) #вернёт одно число — среднюю ошибку по батчу.
        loss.backward()
        optimizer.step()

    
    if epoch % 10 == 0:
        model.eval()
        with torch.no_grad():
            val_output = model(X_val)
            val_loss = criterion(val_output, y_val)
        print(f"epoch {epoch}, train_loss: {loss.item():.4f}, val_loss: {val_loss.item():.4f}")

epoch 0, train_loss: 5.6600, val_loss: 11.2118
epoch 10, train_loss: 2.0838, val_loss: 5.3431
epoch 20, train_loss: 0.7870, val_loss: 2.4517
epoch 30, train_loss: 0.1604, val_loss: 1.2570
epoch 40, train_loss: 0.0962, val_loss: 0.9097
epoch 50, train_loss: 0.4808, val_loss: 0.7787
epoch 60, train_loss: 0.0789, val_loss: 0.6900
epoch 70, train_loss: 0.0873, val_loss: 0.6240
epoch 80, train_loss: 0.2009, val_loss: 0.5520
epoch 90, train_loss: 0.1549, val_loss: 0.4914
epoch 100, train_loss: 0.1028, val_loss: 0.4434
epoch 110, train_loss: 0.0922, val_loss: 0.3890
epoch 120, train_loss: 0.0379, val_loss: 0.3545
epoch 130, train_loss: 0.0348, val_loss: 0.3203
epoch 140, train_loss: 0.0304, val_loss: 0.2834
epoch 150, train_loss: 0.0282, val_loss: 0.2607
epoch 160, train_loss: 0.0479, val_loss: 0.2349
epoch 170, train_loss: 0.0092, val_loss: 0.2221
epoch 180, train_loss: 0.0179, val_loss: 0.2063
epoch 190, train_loss: 0.0188, val_loss: 0.1919


In [7]:
model.eval()
with torch.no_grad():
    test_output = model(X_test)
    test_loss = criterion(test_output, y_test)
    print("test loss: ", test_loss.item())

test loss:  0.05262432619929314
